# FPL next-gameweek forecasting

This short walkthrough reads the committed evaluation report and makes the model comparison easy to inspect. It does not download data, train a model, or generate predictions.

The task is to estimate a player's points in the next gameweek from information available at the current gameweek. The evaluation uses expanding time windows so training rows precede each held-out gameweek.

In [1]:
import json
from pathlib import Path

report_name = Path("reports/retained/2024-25-gw01-15-evaluation.json")
candidates = (
    Path.cwd() / report_name,
    Path.cwd().parent / report_name,
)
report_path = next((path for path in candidates if path.is_file()), None)
if report_path is None:
    raise FileNotFoundError(report_name)
report = json.loads(report_path.read_text(encoding="utf-8"))
data = report["data"]
print(f"Loaded: {report_name}")
print(f"Evaluation rows: {data['evaluated_rows']}")
print(f"Folds: {data['folds']}")
print(
    f"Held-out gameweeks: GW{data['evaluated_gameweek_start']} to GW{data['evaluated_gameweek_end']}"
)

Loaded: reports/retained/2024-25-gw01-15-evaluation.json
Evaluation rows: 6684
Folds: 10
Held-out gameweeks: GW6 to GW15


## Results

Ridge regression is compared with three simple baselines. MAE measures point error, while NDCG@10 measures how well the ranked shortlist prioritises higher-scoring players. Lower MAE is better. Higher NDCG@10 is better.

In [2]:
model_order = (
    ("last_gameweek", "Last gameweek"),
    ("rolling_3_mean", "Three-observation mean"),
    ("training_mean", "Training-window mean"),
    ("ridge_regression", "Ridge regression"),
)
print(f"{'Candidate':<27} {'MAE':>7} {'NDCG@10':>10} {'Spearman':>10} {'R2':>7}")
for key, label in model_order:
    metrics = report["models"][key]
    print(
        f"{label:<27} {metrics['mae']:7.3f} {metrics['ndcg_at_10']:10.3f} "
        f"{metrics['spearman']:10.3f} {metrics['r2']:7.3f}"
    )

Candidate                       MAE    NDCG@10   Spearman      R2
Last gameweek                 1.235      0.084      0.655  -0.274
Three-observation mean        1.130      0.151      0.692   0.111
Training-window mean          1.501      0.002      0.000  -0.000
Ridge regression              1.110      0.226      0.717   0.273


### How to read the comparison

Ridge has the lowest MAE at 1.110 and the strongest NDCG@10 at 0.226 in this evaluation. The three baselines show what can be achieved by carrying forward the last score, averaging recent scores, or using the training-window mean. These are aggregate out-of-fold results for the stated gameweek window, not a live-season claim.

The chart is [model-comparison.svg](../assets/model-comparison.svg). The exact values are also available in [model-comparison.csv](../results/model-comparison.csv).

## Evaluation design

The first training window covers target gameweeks 2 to 5. Each fold then tests one later gameweek, moving forward one gameweek at a time through GW15. Features are built from the current snapshot and the target is the next completed gameweek. The retained report records 18 numeric, as-of features, a ridge alpha of 4.0, and a top-k ranking cutoff of 10.

The source is the public [Vaastav Fantasy Premier League dataset](https://github.com/vaastav/Fantasy-Premier-League/tree/a59a43a8343d960a58cbe7a1f9fba2d2ce431856/data/2024-25/gws), pinned by the source manifest. The raw source files are not included here.

## Code to inspect

- [data.py](../src/fpl_forecasting/data.py): validates completed gameweek snapshots.
- [features.py](../src/fpl_forecasting/features.py): creates features using only information available at the as-of gameweek.
- [evaluation.py](../src/fpl_forecasting/evaluation.py): runs the expanding-window evaluation.
- [models.py](../src/fpl_forecasting/models.py): fits and serialises the ridge model.
- [metrics.py](../src/fpl_forecasting/metrics.py): computes point-error and ranking metrics.
- [operational.py](../src/fpl_forecasting/operational.py): connects validated inputs to repeatable training and prediction commands.

Together these files demonstrate temporal validation, feature-contract design, model comparison, ranking evaluation, reproducible artifacts, and operational checks.